# IELTS Listening — Evaluator QLoRA fine-tune (Kaggle)

Fine-tunes the doc's **separate evaluator** (a Qwen2.5-**7B**-Instruct
LoRA — the doc permits 7B here for efficiency) to judge one answer at a
time:

> Input: Question + Official Answer + Accepted Variants + Student Answer
> Output: verdict / reason / correct_answer / skill

> ⚠️ **Do a clean Restart & Run All on a single T4** (not *T4 x2*, not
> *P100* — see the generator notebook for why). The load cell hard-fails if
> the kernel is dirty, so re-running cells without a restart won't silently
> OOM at train time.

### Before you run
1. Notebook settings: **Accelerator = GPU T4** (a *single* T4) and
   **Internet = On** (for the clone + the Unsloth install). Unlike the
   generator, **7B fits comfortably here** — the evaluator's prompts are
   short (<=1024 tokens), so there's no O(seq^2) attention blow-up.
2. **Run cell 0 first** to `git clone` this repo — it brings
   `backend/data/datasets/evaluator_sft.jsonl` (5989 records) with it. The
   data cell also accepts a Kaggle Dataset named `ielts-listening-sft`.

## 0. Get the repo + datasets

In [1]:
# Cell 0 — pull this repo so the SFT datasets are on the Kaggle filesystem.
# This is a PRIVATE repo, so give Kaggle a token: Add-ons -> Secrets, add
# GITHUB_TOKEN = a PAT with read access to the repo (fine-grained 'Contents:
# read', or a classic token with the 'repo' scope).
# Alternative: skip cloning and add the 'ielts-listening-sft' Kaggle Dataset;
# the data cell below finds the jsonl either way.
import os
REPO_URL = "https://github.com/asiralabi/IELTS.git"
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{_tok}@")
except Exception:
    print("No GITHUB_TOKEN secret - trying anonymous clone "
          "(only works if the repo is public).")
if not os.path.isdir("ielts"):
    !git clone --depth 1 $REPO_URL ielts
!ls -lh ielts/backend/data/datasets/*.jsonl

No GITHUB_TOKEN secret - trying anonymous clone (only works if the repo is public).
Cloning into 'ielts'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 223 (delta 13), reused 218 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (223/223), 1.37 MiB | 5.42 MiB/s, done.
Resolving deltas: 100% (13/13), done.
-rw-r--r-- 1 root root 1.7M Jul 22 09:21 ielts/backend/data/datasets/cambridge_listening.jsonl
-rw-r--r-- 1 root root  11M Jul 22 09:21 ielts/backend/data/datasets/evaluator_sft.jsonl
-rw-r--r-- 1 root root 4.9M Jul 22 09:21 ielts/backend/data/datasets/generator_sft.jsonl


In [2]:
%%capture
# Unsloth gives ~2x faster QLoRA. Two Kaggle-specific gotchas: (1) the free
# Unsloth build trains on ONE GPU only, and (2) it needs a Turing-or-newer
# card — use a **T4** (compute capability 7.5). The P100 is Pascal (6.0) and
# has NO compiled Unsloth/Triton kernels -> 'no kernel image is available'.
# Fit depends on the model AND the corpus length: the generator's records
# are long (~5-7k tokens each), so it uses a 3B base; a 7B/14B OOMs a T4 at
# train time on that corpus (see the generator's section 2 for the math).
# If this install ever breaks, follow the current Kaggle snippet at
# https://github.com/unslothai/unsloth (the API used below is stable).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 1. Load Qwen2.5-7B in 4-bit

In [3]:
import os
# Set BEFORE the torch/unsloth CUDA import. Lets the allocator grow
# segments instead of failing on a large contiguous request — cheap VRAM
# hygiene that reduces fragmentation-driven OOMs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Pin to ONE GPU. Unsloth's free build trains on a single GPU anyway, but
# if TWO are visible (e.g. you picked 'T4 x2') it loads under an accelerate
# device_map dispatch whose per-forward hooks pile extra tensors onto GPU 0
# and OOM it. One visible GPU = clean single-device load. Set before import.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc, sys, torch
# ---- Dirty-kernel guard (the #1 cause of OOM in this notebook) ----------
# If you re-run cells WITHOUT restarting, a previous run's model+optimizer
# stay resident on the GPU and the next load stacks on top -> a misleading
# CUDA OOM at trainer.train() even though the model fits fresh. Popping the
# global names is NOT enough: IPython pins those GPU tensors through the
# stored traceback of the previous OOM (sys.last_traceback holds every
# frame's locals) and through the Out[]/_ output cache. Clear all of them,
# then VERIFY the GPU is actually clean and fail fast with an actionable
# message if it isn't — far better than a cryptic OOM five cells later.
for _n in ("model", "tokenizer", "trainer", "trainer_stats", "dataset", "raw"):
    globals().pop(_n, None)
try:
    _ip = get_ipython()
    _ip.user_ns.get("Out", {}).clear()
    for _v in ("_", "__", "___", "_i", "_ii", "_iii"):
        _ip.user_ns.pop(_v, None)
except Exception:
    pass
sys.last_type = sys.last_value = sys.last_traceback = None
for _ in range(3):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

if torch.cuda.is_available():
    _free, _total = torch.cuda.mem_get_info()
    _used = (_total - _free) / 1024**3
    if _used > 2.0:
        raise RuntimeError(
            f"{_used:.1f} GiB is STILL resident on the GPU before loading — "
            "this kernel is DIRTY (a previous run's model was not freed). "
            "Fix: kernel menu -> 'Restart & Clear Cell Outputs', then Run "
            "All. Re-running cells without a restart stacks models on the "
            "GPU and causes the misleading OOM at trainer.train()."
        )
    print(f"GPU clean: {_used:.2f} GiB resident before load — good to go.")

from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,   # QLoRA
    # Pin the whole model to GPU 0: no cross-GPU sharding, and the model
    # device matches the trainer device. A bnb 4-bit model loaded on a
    # different device than the trainer makes accelerate raise ValueError
    # ('can't train a model loaded in 4-bit precision on a different
    # device...') at trainer.train(); this is the fix it recommends.
    device_map={"": 0},
)

GPU clean: 0.10 GiB resident before load — good to go.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 2. Attach LoRA adapters

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",   # long scripts -> save VRAM
    random_state=3407,
)

Unsloth 2026.7.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 3. Load the SFT dataset

Evaluator prompts are short, so `MAX_SEQ_LEN=1024` is plenty and keeps
training fast.

In [5]:
import glob, os
from datasets import load_dataset

# Resolves the SFT jsonl produced by backend/tools/build_dataset.py.
# Works whether you git-clone this repo into the notebook OR add it as a
# Kaggle Dataset named 'ielts-listening-sft'.
FILENAME = "evaluator_sft.jsonl"
CANDIDATES = [
    f"/kaggle/input/ielts-listening-sft/{FILENAME}",
    *glob.glob(f"/kaggle/**/backend/data/datasets/{FILENAME}", recursive=True),
    *glob.glob(f"**/backend/data/datasets/{FILENAME}", recursive=True),
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"{FILENAME} not found. Git-clone this repo (see cell 0) or add the "
        "Kaggle Dataset 'ielts-listening-sft'.")
print("Using dataset:", DATA_PATH)

raw = load_dataset("json", data_files=DATA_PATH, split="train")

def to_text(row):
    # Each row is {"messages": [system, user, assistant]}; render with
    # the model's own chat template so training matches inference.
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset)
print(dataset[0]["text"][:600])

# Guard against SILENT truncation. SFTTrainer cuts any sample longer than
# MAX_SEQ_LEN, and because the assistant JSON (the training target) comes
# LAST, truncation quietly destroys the label -> the model learns to emit
# incomplete exams. Measure real token lengths and fail fast if any sample
# would be cut, instead of training on corrupted targets.
_lens = sorted(len(tokenizer(t, add_special_tokens=False)["input_ids"])
               for t in dataset["text"])
_over = [n for n in _lens if n > MAX_SEQ_LEN]
print(f"token lengths: min={_lens[0]} median={_lens[len(_lens)//2]} "
      f"max={_lens[-1]} | MAX_SEQ_LEN={MAX_SEQ_LEN} | over={len(_over)}")
assert not _over, (
    f"{len(_over)} sample(s) exceed MAX_SEQ_LEN={MAX_SEQ_LEN} (max is "
    f"{_lens[-1]}); they would be truncated and corrupt the target. "
    "Raise MAX_SEQ_LEN (costs VRAM) or shorten those records.")

Using dataset: /kaggle/working/ielts/backend/data/datasets/evaluator_sft.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5989 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 5989
})
<|im_start|>system
You are an IELTS Listening answer evaluator. You judge ONE answer at a time. You are given the question, the official answer, the list of accepted variant forms, and the student's answer. Decide whether the student's answer is correct under official IELTS clerical-marking rules.

Marking rules:
- Ignore case and leading/trailing whitespace.
- Accept any form listed in Accepted Variants as fully correct, in addition to the official answer.
- Accept numbers written as digits or words ("20" = "twenty"), standard abbreviations, and both British and American spellings.
- Reject: 
token lengths: min=376 median=399 max=579 | MAX_SEQ_LEN=1024 | over=0


## 4. Train (response-only loss)

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="qwen2.5-7b-ielts-evaluator-lora-checkpoints",
        report_to="none",
    ),
)

# Mask the prompt: only the assistant JSON contributes to the loss, so
# the model learns to PRODUCE exams, not to echo the spec/instructions.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/5989 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/5989 [00:00<?, ? examples/s]

In [7]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,989 | Num Epochs = 2 | Total steps = 1,498
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.043366
2,2.082767
3,2.051080
4,1.838717
5,1.334388
6,1.126009
7,0.725953
8,0.730114
9,0.607811
10,0.472720


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-7b-ielts-evaluator-lora-checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-7b-ielts-evaluator-lora-checkpoints/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-7b-ielts-evaluator-lora-checkpoints/checkpoint-1498/tokenizer_config.json.


TrainOutput(global_step=1498, training_loss=0.013519680354097417, metrics={'train_runtime': 20967.2309, 'train_samples_per_second': 0.571, 'train_steps_per_second': 0.071, 'total_flos': 2.051292864854016e+17, 'train_loss': 0.013519680354097417, 'epoch': 2.0})


## 5. Save adapter + GGUF

In [8]:
# 1) LoRA adapter only (tiny, ~100-300 MB) — load on top of the base.
model.save_pretrained("qwen2.5-7b-ielts-evaluator-lora")
tokenizer.save_pretrained("qwen2.5-7b-ielts-evaluator-lora")

# 2) GGUF (q4_k_m) for llama.cpp / Ollama serving on CPU or small GPU.
#    Merges + quantises; needs several GB of /kaggle/working disk.
model.save_pretrained_gguf("qwen2.5-7b-ielts-evaluator-gguf", tokenizer, quantization_method="q4_k_m")

# 3) (optional) merged 16-bit HF weights for vLLM (~6 GB for 3B, ~14 GB for
#    7B) — uncomment
#    only if you have the disk / will push to the HF Hub.
# model.save_pretrained_merged("qwen2.5-7b-ielts-evaluator-lora-merged16", tokenizer, save_method="merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-7b-ielts-evaluator-lora/tokenizer_config.json.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-7b-ielts-evaluator-gguf/tokenizer_config.json.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:49<00:00, 12.26s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:04<00:00, 31.06s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen2.5-7b-ielts-evaluator-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10079-mix-fb3d4ca (app-b10079-mix-fb3d4ca-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed in Kaggle environment.
This is likely due to the 20GB disk space limit.
Try saving to /tmp directory or use a smaller model.
Error: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile Qwen2.5-7B-Instruct.F16.gguf --outtype f16 --split-max-size 50G qwen2.5-7b-ielts-evaluator-gguf`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'Qwen2.5-7B-Instruct.F16.gguf', '--outtype', 'f16', '--split-max-size', '50G', 'qwen2.5-7b-ielts-evaluator-gguf']' returned non-zero exit status 1.
--- converter stderr ---
INFO:hf-to-gguf:Loading model: qwen2.5-7b-ielts-evaluator-gguf
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-0000

## 6. Serve it as a second model

Unsloth writes the GGUF to a sibling `qwen2.5-7b-ielts-evaluator-gguf_gguf/`
folder containing `Qwen2.5-7B-Instruct.Q4_K_M.gguf` + a ready `Modelfile`.
Download that folder and build its own Ollama model:
```
ollama create ielts-evaluator -f Modelfile
```
The evaluator is a **separate** model from the generator, so using it
for marking needs a small backend addition: a second client pointed at
`ielts-evaluator` that runs the `EVALUATOR_SYSTEM` prompt
(`app/llm/prompts.py`) per answer, feeding results into
`listening_trainer.check_full_test`. Until that wiring lands, the base
model + `ANSWER_CHECKER_SYSTEM` continues to mark whole sets. The
system prompt this model was trained on lives in `EVALUATOR_SYSTEM`.